MODELO BOOLEANO

É o metodo mais simples de busca, ele trata os documentos como CONJUNTOS DE TERMOS e usa lógica booleana para responser consultas.
Operadores:
1. AND
2. OR
3. NOT

Exemplo de consulta:
- Movie AND horror

Essa consulta vai retornar os documentos que contem ambas as palavras.

In [2]:
#exemplo em código:
docs = [
"this movie is amazing",
"this horror movie is terrible",
"amazing acting but boring story"
]

def boolean_search(query, docs):
    q = query.split(" AND ")
    results = []

    for doc in docs:
        if all(term in doc for term in q):
            results.append(doc)

    return results

print(boolean_search("movie AND horror", docs))

['this horror movie is terrible']


Um ponto fraco inato dessa abordagem é que ela nao ranqueia documentos, todos os resultados sao igualmente relevantes.
O problema disso é que o usuario nao tem como saber qual dos achados é o documento mais relevante.

Onde é usado:
1. Sitemas jurídicos
2. bases academicas
3. buscas avançadas

Um esemplo de uso é o Google Scholar: deep learning AND computer vision

MODELO DE ESPAÇO VETORIAL

Foi criado justamente pra resolver o problema do modelo booleano, ele representa documentos como vetores numéricos, onde cada palavra é uma dimensão.

Representacao:
- [movie, horror, amazing]
1. Docucumento:
- this horror movie is amazing
2. vetor:
- [1, 1, 1]
3. Outro documento:
- amazing acting
4. Vetor:
- [0, 0, 1]

Como documentos e consultas são representados como vetores, a ideia é comparar a direção desses vetores. A similaridade do cosseno mede, de forma trigonométrica, o quão alinhada está a direção do vetor da consulta em relação aos vetores da base. Quanto mais alinhados estiverem, mais semelhantes são os textos.

A mais usada é a similaridade de cosseno:

Formula:

similarity(A, B) = A * B / ||A|| * ||B||

o retorno disso na pratica pode ser interpretado assim:

1 -> texto iguais
0 -> sem relacao

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
"this movie is amazing",
"this horror movie is terrible",
"amazing acting but boring story"
]

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(docs).toarray()

query = ["amazing movie"]
q_vec = vectorizer.transform(query).toarray()[0]

print("VOCABULARIO:", vectorizer.get_feature_names_out())
print("\nVETORES DOS DOCUMENTOS:\n", X)
print("\nVETOR DA QUERY:\n", q_vec)

VOCABULARIO: ['acting' 'amazing' 'boring' 'but' 'horror' 'is' 'movie' 'story'
 'terrible' 'this']

VETORES DOS DOCUMENTOS:
 [[0.         0.5        0.         0.         0.         0.5
  0.5        0.         0.         0.5       ]
 [0.         0.         0.         0.         0.51741994 0.3935112
  0.3935112  0.         0.51741994 0.3935112 ]
 [0.46735098 0.35543247 0.46735098 0.46735098 0.         0.
  0.         0.46735098 0.         0.        ]]

VETOR DA QUERY:
 [0.         0.70710678 0.         0.         0.         0.
 0.70710678 0.         0.         0.        ]


In [15]:
import numpy as np

def cosine_sim_debug(a, b):

    print("\n--- NOVO CALCULO ---")

    print("vetor A:", a)
    print("vetor B:", b)

    # produto escalar
    dot = np.dot(a, b)
    print("produto escalar (A · B):", dot)

    # norma dos vetores
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    print("||A||:", norm_a)
    print("||B||:", norm_b)

    # similaridade final
    sim = dot / (norm_a * norm_b)

    print("similaridade cosseno:", sim)

    return sim

In [16]:
scores = []

for i, doc_vec in enumerate(X):
    print(f"\nComparando QUERY com DOC {i}")
    score = cosine_sim_debug(q_vec, doc_vec)
    scores.append(score)

print("\nScores finais:", scores)


Comparando QUERY com DOC 0

--- NOVO CALCULO ---
vetor A: [0.         0.70710678 0.         0.         0.         0.
 0.70710678 0.         0.         0.        ]
vetor B: [0.  0.5 0.  0.  0.  0.5 0.5 0.  0.  0.5]
produto escalar (A · B): 0.7071067811865476
||A||: 1.0
||B||: 1.0
similaridade cosseno: 0.7071067811865476

Comparando QUERY com DOC 1

--- NOVO CALCULO ---
vetor A: [0.         0.70710678 0.         0.         0.         0.
 0.70710678 0.         0.         0.        ]
vetor B: [0.         0.         0.         0.         0.51741994 0.3935112
 0.3935112  0.         0.51741994 0.3935112 ]
produto escalar (A · B): 0.2782544408877314
||A||: 1.0
||B||: 1.0
similaridade cosseno: 0.2782544408877314

Comparando QUERY com DOC 2

--- NOVO CALCULO ---
vetor A: [0.         0.70710678 0.         0.         0.         0.
 0.70710678 0.         0.         0.        ]
vetor B: [0.46735098 0.35543247 0.46735098 0.46735098 0.         0.
 0.         0.46735098 0.         0.        ]
produto 

Uso do TF-IDF funciona aqui porque ele penaliza palavras comuns.

Exemplo:

the
is
movie

aparecem em quase todos os documentos. Já palavras raras têm mais peso.

Onde é usado:

1. motores de busca antigos
2. sistemas de recomendação textual
3. recuperação científica
4. baseline de NLP

MODELO PROBABILÍSTICO

Em vez de medir a distancia vetorial, é estimada:

P = (documento relevante | consulta)

Ou seja, qual a probabilidade desse documento ser relevante?

Um dos algoritmos mais utilizados do modelo probabilistico é o BM25

Alem do TF-IDF base, ele considera:
1. frequencia da palavra
2. tamanho do documento
3. raridade da palavra

A intuicao que da pra formar a partir disso é mais ou menos assim:

- 1 vez -> Util
- 5 vezes -> mais Util
- 50 vezes -> saturacao

Ou seja: Mais ocorrencias ajudam, mas com retorno crescente.

In [21]:
from rank_bm25 import BM25Okapi

docs = [
"this movie is amazing",
"this horror movie is terrible",
"amazing acting but boring story"
]

tokenized = [doc.split() for doc in docs]

print(tokenized)

bm25 = BM25Okapi(tokenized)

query = "amazing movie".split()

scores = bm25.get_scores(query)

print(scores)

[['this', 'movie', 'is', 'amazing'], ['this', 'horror', 'movie', 'is', 'terrible'], ['amazing', 'acting', 'but', 'boring', 'story']]
[0.05459205 0.02474588 0.02474588]


Vantagem sobre TF-IDF por exemplo.

Trata melhor:
1. documentos muito longos
2. repeticao de termos
3. distribuicao de palavras

Por isso ele virou padrao em motores de busca modernos

Arquitetura moderna de busca

Hoje muitos sistemas usam combinação de modelos:


consulta

   ↓

BM25

   ↓

top 100 documentos

   ↓

reranker neural

   ↓

top 5

Esse pipeline aparece em:

1. Elasticsearch
2. Bing
3. sistemas RAG